# How to Access GES DISC Data Using Python

<p></p>

<div style="background:#FFFFC5; border:1px solid #000000; padding:5px 10px; color:#000000;">
    Please, be very judicious when working on long data time series residing on a remote data server.<br />
    It is very likely that attempts to apply similar approaches on remote data, such as hourly data, for more than a year of data at a time, will result in a heavy load on the remote data server. This may lead to negative consequences, ranging from very slow performance that will be experienced by hundreds of other users, up to denial of service.
</div>

### Overview

There are multiple ways to work with GES DISC data resources using Python. For example, the data can accessed using [techniques that rely on a native Python code](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html). 

Still, there are several third-party libraries that can further simplify the access. In the sections below, we demonstrate downloading and streaming granules to the notebook using these libraries.

The examples will use a sample MERRA-2 granule, from the [M2T1NXSLV.5.12.4 collection](https://disc.gsfc.nasa.gov/datasets/M2T1NXSLV_5.12.4/summary?keywords=M2T1NXSLV_5.12.4), to demonstrate data access.

### Prerequisites

<div style="background:#ADD8E6; border:1px solid #000000; padding:5px 10px; color:#000000;">
    <strong>Warning:</strong> An Earthdata Login account with the "NASA GES DISC DATA ARCHIVE" and "Hyrax in the Cloud" applications enabled are required to access GES DISC data and store "Earthdata prerequisite files". To create an Earthdata Login account, and enable these applications, please visit <a href="https://disc.gsfc.nasa.gov/earthdata-login" target="_blank">this guide</a>.
</div>

This notebook was written using Python 3.10, and requires these libraries and files:

- `netrc` or `edl_token` file with valid Earthdata Login credentials
   - [How to Generate Earthdata Prerequisite Files](https://disc.gsfc.nasa.gov/information/howto?title=How%20to%20Generate%20Earthdata%20Prerequisite%20Files)
- [requests](https://docs.python-requests.org/en/latest/) (version 2.22.0 or later)
- [pydap](https://github.com/pydap/pydap) (we recommend using version 3.5)
- [xarray](https://docs.xarray.dev/en/stable/)
- [netCDF4-python](https://github.com/Unidata/netcdf4-python) (we recommend using version 1.6.2)
- [earthaccess](https://earthaccess.readthedocs.io/en/latest/quick-start/)
- ***Optional:***
   - For OPeNDAP examples, this notebook can be run using the ['opendap' YAML file](https://github.com/nasa/gesdisc-tutorials/tree/main/environments/opendap.yml) provided in the 'environments' subfolder. Please follow the instructions [here](https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html#creating-an-environment-from-an-environment-yml-file) to install and activate this environment.
 

### Contents
* [Download Full Granule Data](#download_full_granules)
    * [Option 1: Use `requests`](#download_requests)
    * [Option 2: Use `earthaccess`](#download_earthaccess)
* [Stream Full Granule Data](#stream_full_granules)

* [Subset and Stream Granule Data from OPeNDAP Servers](#opendap)
    * [Option 1a: Use `pydap` with Earthdata Login credentials ](#opendap_pydap)
    * [Option 1b: Use `pydap` with Earthdata Login credentials ](#opendap_pydap_token)
    * [Option 2: Use `xarray`](#opendap_xarray)
    * [Option 3: Use `netcdf4-python`](#opendap_netcdf4-python)
* [Subset and Stream Granule Data from THREDDS Servers](#thredds)
    * [Option 1: Use `xarray`](#thredds_xarray)
* [Subset and Stream Data from the Level 3/4 Subsetter and Regridder API](#l34rs)

### Links Used in this Notebook

There are several example links that will be used to access data from the same granule. Each link can be searched for using several tools, including [Earthdata Search](https://search.earthdata.nasa.gov/search/granules/granule-details?p=C1276812863-GES_DISC&pg[0][v]=f&pg[0][qt]=1980-01-01%2C1981&pg[0][gsk]=-start_date&g=G1277898447-GES_DISC&q=M2T1NXSLV&tl=1723660883!3!!), the [dataset landing page](https://disc.gsfc.nasa.gov/datasets/M2T1NXSLV_5.12.4/summary?keywords=M2T1NXSLV_5.12.4) for the particular collection, or through the [Content Metadata Repository](https://cmr.earthdata.nasa.gov/virtual-directory/collections/C1276812863-GES_DISC/temporal/1980/01/01). Links can be generated programmatically using the `earthaccess` library, the [python-cmr](https://github.com/nasa/python_cmr) library, or the [Web Services API](https://disc.gsfc.nasa.gov/information/howto?keywords=dataset%).

Links used in this notebook:
- HTTPS: https://data.gesdisc.earthdata.nasa.gov/data/MERRA2/M2T1NXSLV.5.12.4/1980/01/MERRA2_100.tavg1_2d_slv_Nx.19800101.nc4
- OPeNDAP ([?](https://disc.gsfc.nasa.gov/information/documents?title=OPeNDAP%20In%20The%20Cloud)): 
    - OPeNDAP Subsetting Page: https://opendap.earthdata.nasa.gov/collections/C1276812863-GES_DISC/granules/M2T1NXSLV.5.12.4%3AMERRA2_100.tavg1_2d_slv_Nx.19800101.nc4.dmr.html
    - Example OPeNDAP URL (No Subsetting): https://opendap.earthdata.nasa.gov/collections/C1276812863-GES_DISC/granules/M2T1NXSLV.5.12.4%3AMERRA2_100.tavg1_2d_slv_Nx.19800101.nc4
    - Example OPeNDAP URL (With variable subset): https://opendap.earthdata.nasa.gov/collections/C1276812863-GES_DISC/granules/M2T1NXSLV.5.12.4%3AMERRA2_100.tavg1_2d_slv_Nx.19800101.nc4?dap4.ce=/T2M[0:1:23][0:1:360][0:1:575]
- THREDDS ([?](https://docs.unidata.ucar.edu/tds/4.6/adminguide/#:~:text=Overview,other%20remote%20data%20access%20protocols.)): 
    - Example THREDDS URL Subsetting Page: https://goldsmr4.gesdisc.eosdis.nasa.gov/thredds/dodsC/MERRA2_aggregation/M2T1NXSLV.5.12.4/M2T1NXSLV.5.12.4_Aggregation_1980.ncml.html
    - Example THREDDS URL (for `Xarray` access only): https://goldsmr4.gesdisc.eosdis.nasa.gov/thredds/dodsC/MERRA2_aggregation/M2T1NXSLV.5.12.4/M2T1NXSLV.5.12.4_Aggregation_1980.ncml


---



#### Option 2: Use `earthaccess`  <a class="anchor" id="download_earthaccess"></a>

The `earthaccess` library can be used to search for granules and download them to your local machine, based on the collection shortname, longname, version, or DOI, gathered from the collection dataset landing page or the Content Metadata Repository. The `search_data` function will search for granules inside the specified temporal and bounding box ranges, and will return a list of URLs to be downloaded. Finally, it will download these URLs, assuming you have been authenticated using your previously-generated Earthdata prerequisite files.

Please note that as of August 2024, `earthaccess` does not have the ability to return OPeNDAP URLs.

In [3]:
import earthaccess
from pathlib import Path

out_dir = Path('data/ECOCO_V1')
out_dir.mkdir(parents=True, exist_ok=True)

# This will work if Earthdata prerequisite files have already been generated
auth = earthaccess.login()

# To download multiple files, change the second temporal parameter
results = earthaccess.search_data(
    doi="10.5067/JY21I1T22RCA", #ECOCO3.0 ECOSTRESS/OCO-3 Co-Located Observations V1.0 (ECOCO3)
    temporal=('2019-08-05', '2023-01-29'), # This will stream one granule, but can be edited for a longer temporal extent
    bounding_box=(-180, -90, 180, 90)
)

downloaded_files = earthaccess.download(
    results,
    local_path=str(out_dir), 
)

QUEUEING TASKS | : 100%|██████████| 5227/5227 [00:00<00:00, 192870.83it/s]
PROCESSING TASKS | : 100%|██████████| 5227/5227 [17:34<00:00,  4.96it/s]  
COLLECTING RESULTS | : 100%|██████████| 5227/5227 [00:00<00:00, 2010419.72it/s]
